# [NEXON Open API] 마비노기 영웅전 거래소·공지 수집

**데이터 출처**
  * Data based on NEXON Open API (https://openapi.nexon.com)

**수집 내용**
| 데이터 | 엔드포인트 | 수집 주기 |
| :--- | :--- | :--- |
| 인챈트 스크롤 시세 (최근 1주) | `/heroes/v2/marketplace/market-history` | 하루 1회 |
| 골드 구매 상위 30명 (최근 1주) | `/heroes/v2/marketplace/gold-market-buy-top-30` | 하루 1회 |
| 골드 판매 상위 30명 (최근 1주) | `/heroes/v2/marketplace/gold-market-sell-top-30` | 하루 1회 |
| 공지사항 목록 (최근 20개) | `/heroes/v1/notice` | 하루 1회 |
| 패치노트 목록 (최근 20개) | `/heroes/v1/notice-patch` | 하루 1회 |
| 이벤트&샵 목록 (최근 20개) | `/heroes/v1/notice-event` | 하루 1회 |

* 인챈트 스크롤 시세는 "인챈트 스크롤", "인챈트 스크롤 (무기한)"으로 검색
* 수집한 원본 데이터는 BigQuery에만 저장하며, 외부에 공개하지 않습니다.

In [ ]:
# 환경 설정
import os
import json
import time
from datetime import datetime
from zoneinfo import ZoneInfo

import requests
import pandas as pd
from dotenv import load_dotenv

from google.cloud import bigquery

In [ ]:
# .env에서 API 키 불러오기
load_dotenv("../.env", override=True)

GCP_PROJECT_ID = os.getenv("GCP_PROJECT_ID")
HEROES_KEY = os.getenv("NEXON_HEROES_API_KEY")

print("API 키 불러오기: ", "성공" if HEROES_KEY else "실패")

# api 키는 헤더에 넣어서 보내기
headers = {
    "x-nxopen-api-key": HEROES_KEY
}

# 수집 시각 (한국 시간)
collected_at = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
# 마비노기 영웅전 거래소 거래 기록 api (최근 1주 데이터 제공, item_name 필수)
API_URL = "https://open.api.nexon.com/heroes/v2/marketplace/market-history"

# 영웅전 키로 headers 다시 만들기 (다른 게임 키와 섞이지 않도록)
headers = {
    "x-nxopen-api-key": HEROES_KEY
}

# 이 두 이름으로 검색하면 모든 인챈트 스크롤이 조회됨
item_names = ["인챈트 스크롤", "인챈트 스크롤 (무기한)"]

enchant_price = []
failed_count = 0
request_count = 0

for item_name in item_names:
    cursor = ""

    # 1주치 기록이 500건을 넘으면 다음 페이지까지 받기 (최대 20페이지)
    for page in range(20):
        params = {
            "item_name": item_name
        }

        # 첫 페이지에는 cursor를 넣지 않음 (빈 값을 보내면 400 에러)
        if cursor:
            params["cursor"] = cursor

        try:
            response = requests.get(API_URL, params=params, headers=headers, timeout=30)
            request_count += 1

            # 실패하면 넥슨이 보낸 에러 내용 출력
            if response.status_code != 200:
                print(f"{item_name} 에러 내용: {response.text}")

            # HTTP 요청에 문제가 있으면 오류 발생하고 중단
            response.raise_for_status()

            # 가져온 api를 json 형태로 변경
            data = response.json()

            for item in data["item"]:
                option = item.get("item_option") or {}

                # 인챈트 종류를 별도 칸으로 꺼내기 (접두 또는 접미 중 하나에 들어 있음)
                item["prefix_enchant"] = option.get("prefix_enchant_preset_1")
                item["suffix_enchant"] = option.get("suffix_enchant_preset_1")

                # 원본 옵션은 글자로 바꿔서 보관
                item["item_option"] = json.dumps(option, ensure_ascii=False)

                item["search_item_name"] = item_name
                item["collected_at"] = collected_at
                enchant_price.append(item)

            # 다음 페이지 cursor가 없으면 이 아이템은 끝
            cursor = data.get("next_cursor")
            if not cursor:
                break

        except Exception as e:
            print(f"{item_name} {page + 1}페이지 요청 실패: {e}")
            failed_count += 1
            break

        finally:
            time.sleep(1)

enchant_price_df = pd.DataFrame(enchant_price)

print("수집 완료된 데이터 개수:", len(enchant_price_df))
print("수집 실패 횟수:", failed_count)
print("API 요청 횟수:", request_count)

In [ ]:
# 마비노기 영웅전 골드 거래소 구매/판매 상위 30명 api (최근 1주)
gold_apis = {
    "buy": ("https://open.api.nexon.com/heroes/v2/marketplace/gold-market-buy-top-30", "buy_gold"),
    "sell": ("https://open.api.nexon.com/heroes/v2/marketplace/gold-market-sell-top-30", "sell_gold"),
}

gold_top30 = {}

for name, (url, data_key) in gold_apis.items():
    try:
        response = requests.get(url, headers=headers, timeout=30)

        # HTTP 요청에 문제가 있으면 오류 발생하고 중단
        response.raise_for_status()

        # 가져온 api를 json 형태로 변경
        data = response.json()

        records = data[data_key]

        # 순위와 수집 시각 붙이기 (응답이 많은 순서대로 온다고 가정)
        for rank, item in enumerate(records, start=1):
            item["rank"] = rank
            item["collected_at"] = collected_at

        gold_top30[name] = pd.DataFrame(records)
        print(f"골드 {name} 상위 30: {len(records)}명")

    except Exception as e:
        print(f"골드 {name} 요청 실패: {e}")

    time.sleep(1)

gold_buy_df = gold_top30.get("buy")
gold_sell_df = gold_top30.get("sell")

In [ ]:
# 마비노기 영웅전 공지 api (각각 최근 20개)
notice_apis = {
    "notice": ("https://open.api.nexon.com/heroes/v1/notice", "notice"),
    "patch": ("https://open.api.nexon.com/heroes/v1/notice-patch", "patch_notice"),
    "event": ("https://open.api.nexon.com/heroes/v1/notice-event", "event_notice"),
}

notices = {}

for name, (url, data_key) in notice_apis.items():
    try:
        response = requests.get(url, headers=headers, timeout=30)

        # HTTP 요청에 문제가 있으면 오류 발생하고 중단
        response.raise_for_status()

        # 가져온 api를 json 형태로 변경
        data = response.json()

        records = data[data_key]

        for item in records:
            item["collected_at"] = collected_at

        notices[name] = pd.DataFrame(records)
        print(f"{name}: {len(records)}건")

    except Exception as e:
        print(f"{name} 요청 실패: {e}")

    time.sleep(1)

notice_df = notices.get("notice")
notice_patch_df = notices.get("patch")
notice_event_df = notices.get("event")

In [ ]:
# 수집 결과 요약
results = {
    "인챈트 스크롤 시세": enchant_price_df,
    "골드 구매 상위 30": gold_buy_df,
    "골드 판매 상위 30": gold_sell_df,
    "공지사항": notice_df,
    "패치노트": notice_patch_df,
    "이벤트&샵": notice_event_df,
}

for name, df in results.items():
    count = 0 if df is None else len(df)
    print(f"{name}: {count:,}행")

In [ ]:
# BigQuery 연결
client = bigquery.Client(project=GCP_PROJECT_ID)

def upload_to_bigquery(df, table_id):
    # 데이터가 없으면 건너뛰기
    if df is None or df.empty:
        print(f"{table_id}: 업로드할 데이터 없음")
        return

    df = df.copy()

    # 글자 컬럼은 문자열 타입으로 고정 (실행할 때마다 타입이 바뀌지 않도록)
    for col in df.select_dtypes(include=["object", "str"]).columns:
        df[col] = df[col].astype("string")

    # 기존 테이블 아래에 이어 붙이기 (테이블이 없으면 자동으로 생성)
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")

    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()   # 업로드가 끝날 때까지 기다리기

    print(f"{table_id}: {len(df):,}행 업로드 완료")


DATASET = f"{GCP_PROJECT_ID}.mabinogi_heroes"

upload_to_bigquery(enchant_price_df, f"{DATASET}.enchant_price")
upload_to_bigquery(gold_buy_df, f"{DATASET}.gold_buy_top30")
upload_to_bigquery(gold_sell_df, f"{DATASET}.gold_sell_top30")
upload_to_bigquery(notice_df, f"{DATASET}.notice")
upload_to_bigquery(notice_patch_df, f"{DATASET}.notice_patch")
upload_to_bigquery(notice_event_df, f"{DATASET}.notice_event")